In [17]:
import json
import asyncio
import aiohttp
import os
import time
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

API_KEY = os.getenv("RM_Deepseek_R1_PaidKey")
BASE_URL = "https://api.deepseek.com/v1/chat/completions"


In [18]:
INPUT_FILE = "safari_cleaned.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    cleaned_data = json.load(f)

print("Total cleaned items:", len(cleaned_data))

# Choose your dynamic range:
FROM = 4
UPTO = 11

data_chunk = cleaned_data[FROM:UPTO]
print(f"Items selected for enrichment: {len(data_chunk)}")


Total cleaned items: 1101
Items selected for enrichment: 7


In [19]:
def build_prompt(item):
    return [
        {
            "role": "system",
            "content": (
                "You are a travel data enrichment model. "
                "Extract structured fields and return ONLY valid JSON."
            ),
        },
        {
            "role": "user",
            "content": (
                "Extract the following structured features from this safari item:\n"
                "park_names, primary_wildlife_focus, safari_type_detail, suitability,\n"
                "duration_category, price_category, key_attractions, flexibility_notes,\n"
                "district, latitude, longitude.\n\n"
                "Safari Input:\n" + json.dumps(item, indent=2)
            ),
        },
    ]


In [20]:
async def call_deepseek(session, item, retries=5):
    delay = 1

    for attempt in range(retries):
        try:
            payload = {
                "model": "deepseek-reasoner",
                "messages": build_prompt(item),
                "temperature": 0.1,
            }

            async with session.post(
                BASE_URL,
                headers={
                    "Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json",
                },
                json=payload,
                timeout=45,
            ) as resp:
                
                if resp.status == 200:
                    data = await resp.json()
                    result_text = data["choices"][0]["message"]["content"]

                    # Parse JSON (model returns clean JSON most of the time)
                    try:
                        return json.loads(result_text)
                    except:
                        # fallback: extract JSON inside text
                        cleaned = extract_json_safe(result_text)
                        return json.loads(cleaned)

                else:
                    print(f"Error {resp.status}: retrying...")

        except Exception as e:
            print(f"Error: {e} | Retrying... (attempt {attempt + 1})")

        await asyncio.sleep(delay)
        delay *= 2  # exponential backoff

    return None


In [21]:
import re

def extract_json_safe(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group()
    raise ValueError("No JSON object found")


In [22]:
async def enrich_single_item(session, item, index):
    enriched = await call_deepseek(session, item)
    return index, enriched


In [23]:
async def enrich_all_async(items):
    enriched_results = [None] * len(items)
    concurrency_limit = 5  # you can increase to 10 if needed

    connector = aiohttp.TCPConnector(limit_per_host=concurrency_limit)

    async with aiohttp.ClientSession(connector=connector) as session:

        tasks = [
            enrich_single_item(session, item, idx)
            for idx, item in enumerate(items)
        ]

        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Enriching"):
            idx, enriched = await f
            enriched_results[idx] = enriched

    return enriched_results


In [24]:
enriched_data = await enrich_all_async(data_chunk)

print("Enrichment complete!")

Enriching:  14%|█▍        | 1/7 [00:43<04:18, 43.10s/it]

Error:  | Retrying... (attempt 1)
Error:  | Retrying... (attempt 1)
Error:  | Retrying... (attempt 1)
Error:  | Retrying... (attempt 1)
Error:  | Retrying... (attempt 1)
Error:  | Retrying... (attempt 1)
Error:  | Retrying... (attempt 2)
Error:  | Retrying... (attempt 2)
Error:  | Retrying... (attempt 2)
Error:  | Retrying... (attempt 2)
Error:  | Retrying... (attempt 2)
Error:  | Retrying... (attempt 2)


Enriching:  29%|██▊       | 2/7 [02:17<06:06, 73.23s/it]

Error:  | Retrying... (attempt 3)
Error:  | Retrying... (attempt 3)
Error:  | Retrying... (attempt 3)
Error:  | Retrying... (attempt 3)
Error:  | Retrying... (attempt 3)
Error:  | Retrying... (attempt 4)
Error:  | Retrying... (attempt 4)
Error:  | Retrying... (attempt 4)
Error:  | Retrying... (attempt 4)
Error:  | Retrying... (attempt 4)
Error:  | Retrying... (attempt 5)
Error:  | Retrying... (attempt 5)
Error:  | Retrying... (attempt 5)
Error:  | Retrying... (attempt 5)
Error:  | Retrying... (attempt 5)


Enriching: 100%|██████████| 7/7 [04:20<00:00, 37.22s/it]

Enrichment complete!


In [25]:
final_results = []

for i, enriched in enumerate(enriched_data):
    item = data_chunk[i]

    if enriched is None:
        item["extracted_features"] = {}
    else:
        item["extracted_features"] = enriched

    final_results.append(item)


In [26]:
OUTPUT_FILE = f"safari_enriched_{FROM}_{UPTO}.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print("Saved enriched data to:", OUTPUT_FILE)


Saved enriched data to: safari_enriched_4_11.json
